In [1]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn_pandas import DataFrameMapper

import pandas as pd
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index

import torch
import torchtuples as tt

from pycox.datasets import metabric
from pycox.models import CoxPH
from pycox.evaluation import EvalSurv

/home/ykzhou/anaconda3/lib/python3.8/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index


In [2]:
np.random.seed(1234)
_ = torch.manual_seed(123)

In [3]:
df_train = metabric.read_df()
df_test = df_train.sample(frac=0.2)
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
df_train = df_train.drop(df_val.index)

In [4]:
cols_standardize = ['x0', 'x1', 'x2', 'x3', 'x8']
cols_leave = ['x4', 'x5', 'x6', 'x7']

standardize = [([col], StandardScaler()) for col in cols_standardize]
leave = [(col, None) for col in cols_leave]

x_mapper = DataFrameMapper(standardize + leave)

x_train = x_mapper.fit_transform(df_train).astype('float32')
x_val = x_mapper.transform(df_val).astype('float32')
x_test = x_mapper.transform(df_test).astype('float32')

get_target = lambda df: (df['duration'].values, df['event'].values)
y_train = get_target(df_train)
y_val = get_target(df_val)
durations_test, events_test = get_target(df_test)
val = x_val, y_val

In [5]:
in_features = x_train.shape[1]
num_nodes = [32, 32]
out_features = 1
batch_norm = True
dropout = 0.1
output_bias = False

net = tt.practical.MLPVanilla(in_features, num_nodes, out_features, batch_norm,
                              dropout, output_bias=output_bias)

model = CoxPH(net, tt.optim.Adam)
model.optimizer.set_lr(0.01)
epochs = 512
callbacks = [tt.callbacks.EarlyStopping()]
verbose = True

batch_size = 256
callbacks = [tt.cb.EarlyStopping()]

In [6]:
%%time
log = model.fit(x_train, y_train, batch_size, epochs, callbacks, verbose, 
                val_data=val, val_batch_size=batch_size)
_ = model.compute_baseline_hazards()
surv = model.predict_surv_df(x_test)

0:	[0s / 0s],		train_loss: 4.7789,	val_loss: 3.9394
1:	[0s / 0s],		train_loss: 4.6847,	val_loss: 3.9186
2:	[0s / 0s],		train_loss: 4.6303,	val_loss: 3.9229
3:	[0s / 0s],		train_loss: 4.6243,	val_loss: 3.9186
4:	[0s / 0s],		train_loss: 4.6069,	val_loss: 3.9196
5:	[0s / 0s],		train_loss: 4.5824,	val_loss: 3.9125
6:	[0s / 0s],		train_loss: 4.5802,	val_loss: 3.9150
7:	[0s / 0s],		train_loss: 4.5799,	val_loss: 3.9180
8:	[0s / 0s],		train_loss: 4.5664,	val_loss: 3.9291
9:	[0s / 0s],		train_loss: 4.5454,	val_loss: 3.9376
10:	[0s / 0s],		train_loss: 4.5555,	val_loss: 3.9501
11:	[0s / 0s],		train_loss: 4.5432,	val_loss: 3.9564
12:	[0s / 0s],		train_loss: 4.5618,	val_loss: 3.9544
13:	[0s / 0s],		train_loss: 4.5544,	val_loss: 3.9497
14:	[0s / 0s],		train_loss: 4.5439,	val_loss: 3.9382
15:	[0s / 0s],		train_loss: 4.5464,	val_loss: 3.9489
CPU times: user 10.8 s, sys: 808 ms, total: 11.6 s
Wall time: 540 ms


In [7]:
ev = EvalSurv(surv, durations_test, events_test, censor_surv='km')
print('The C-statistics of Cox proportional hazards deep neural network is', ev.concordance_td())

The C-statistics of Cox proportional hazards deep neural network is 0.6601118833700628


In [8]:
cph = CoxPHFitter(penalizer=0.01, l1_ratio = 0)
cph.fit(df_train, duration_col='duration', event_col='event')
print('The C-statistics of Cox proportional hazards model is', 
      concordance_index(df_test['duration'], -cph.predict_partial_hazard(df_test), df_test['event']))

The C-statistics of Cox proportional hazards model is 0.6507270337869346
